# Security of Supply Analysis

:::{note}
**On the workshop JupyterHub, everything is already installed** - just log in and run the cells below, nothing to set up.

No hub access? You can also run this notebook in your browser on [Google Colab](https://colab.research.google.com/github/pypsalabs/workshop-202610/blob/main/berlin/04-pypsa-scenarios.ipynb) (a Google account is required). On Colab, first run this in a cell at the top:

```sh
!pip install pypsa "plotly<6"
```
:::

In this tutorial, we will analyse the security of supply of different scenarios using a pre-built 3-node model of the Philippine power system (Luzon, Visayas, Mindanao). In this setting, we will focus on dispatch optimisation under different stress events and evaluate the performance of the system using different metrics.

We start by importing the necessary libraries and loading the base network. This network is amended by a load shedding generator at each node and we also resample the time series to 4h resolution to speed up the optimisation.

In [ ]:
import pandas as pd
import pypsa

pypsa.options.params.optimize.include_objective_constant = False
pypsa.options.params.optimize.log_to_console = False

pd.options.plotting.backend = "plotly"

In [ ]:
# Backup: https://cloud.pypsalabs.org/s/KPD4AcTtJx2b9Hk/download/phl-network.nc
url = "https://raw.githubusercontent.com/pypsalabs/workshop-202610/main/berlin/data/phl-network.nc"

base = pypsa.Network(url)

In [ ]:
base.add(
    "Generator",
    base.buses.index + " load-shedding",
    bus=base.buses.index,
    carrier="load-shedding",
    marginal_cost=2000,
    p_nom=base.loads_t.p_set.max().max(),
)
base.add("Carrier", "load-shedding", color="crimson")

In [ ]:
base = base.cluster.temporal.resample("8h")

In [ ]:
n0 = base.copy()
n0.name = "base"

## Events

### Power Plant Decommissioning

We decommission 2 GW of coal generation capacity in Luzon and all coal generation capacity in Mindanao.

In [ ]:
n1 = base.copy()
n1.name = "coal-decommissioning"
n1.generators.loc["Luzon coal-subcritical", "p_nom"] -= 2000
n1.generators.loc["Mindanao coal-subcritical", "p_nom"] -= 836

### Power Plant Outage

We simulate the outage of Luzon's supercritical coal power plants in all of February.

In [ ]:
n2 = base.copy()
n2.name = "coal-generator-outage"

n2.generators_t.p_max_pu.loc["2025-02", "Luzon coal-supercritical"] = 0
n2.generators_t.p_max_pu.fillna(1, inplace=True)

### Interconnector Outage

We simulate the outage of the interconnectors between all three regions throughout the full year (i.e. no power can be transmitted between the regions).

In [ ]:
n3 = base.copy()
n3.name = "interconnector-outage"

n3.links.p_max_pu = 0
n3.links.p_min_pu = 0

### Volcano Eruption

We simulate a volcano eruption that reduces solar photovoltaics generation by 80% from February to June.

In [ ]:
n4 = base.copy()
n4.name = "volcano-eruption"

pv_gens = n4.generators.carrier == "photovoltaic"

n4.generators_t.p_max_pu.loc[
    "2025-02":"2025-06",
    pv_gens,
] *= 0.2

### Price Shock in Oil and Gas

We simulate a price shock in oil and gas (e.g. due to a geopolitical crisis) that doubles the fuel costs of oil and gas generators for the full year.

In [ ]:
n5 = base.copy()
n5.name = "price-shock-oil-gas"

selection = n5.generators.carrier.str.contains("oil|gas")
n5.generators.loc[selection, "marginal_cost"] *= 2

### Demand Increase

We simulate a demand increase of 20% across all three regions for the full year (e.g. due to economic growth).

In [ ]:
n6 = base.copy()
n6.name = "demand-increase"

n6.loads_t.p_set *= 1.2

### Drought

We simulate a drought event that reduces hydro generation by 50% for the full year.

In [ ]:
n7 = base.copy()
n7.name = "drought"

# drought reduces hydro generation by 50%
hydro = n7.generators.carrier == "hydro-reservoir-and-run-of-river"
n7.generators_t.p_max_pu.loc[:, hydro] *= 0.5

### Coal Import Limitation

We simulate a coal usage restriction that limits the total coal consumption across the system to 80 TWh thermal for the full year (e.g. due to a coal import limitation).

Note, that we need to factor in the efficiency of the coal generators to calculate the thermal energy consumption from the electrical energy generation.

This is a constraint that is not directly supported by PyPSA, so we need to add it manually to the model as a **custom constraint** after the model has been created but before it is optimised.

In [ ]:
n8 = base.copy()
n8.name = "coal-import-limit"

n8.optimize.create_model()
coal_i = n8.generators.loc[n8.generators.carrier.str.contains("coal")].index
lhs = (
    n8.model["Generator-p"]
    .loc[:, coal_i]
    .mul(n8.snapshot_weightings.generators)
    .sum(dim="snapshot")
    .div(n8.generators.loc[coal_i, "efficiency"])
    .sum()
)
n8.model.add_constraints(lhs <= 80e6, name="Generator-coal_limit");

### Reserve Constraints

In a similar way, we can also add constraints to ensure that there is sufficient reserve capacity available in the system to cover for unexpected events. For example, we can require that there is always at least 20% of the hour's demand or at least 3 GW of reserve capacity available.

In [ ]:
n9 = base.copy()
n9.name = "reserve-constraint"

n9.optimize.create_model()

regex = "coal|gas|geothermal|oil|bioenergy"
reserve_i = n9.generators.loc[n9.generators.carrier.str.contains(regex)].index

We use the dispatch of firm generators (i.e. generators that are not renewable):

In [ ]:
p_reserve = n9.model["Generator-p"].loc[:, reserve_i].sum(dim="name")
p_reserve

as well as the capacity of firm generators to calculate the available reserve capacity in each hour and add a constraint to ensure that this is always above the required reserve margin:

In [ ]:
p_nom_reserve = n9.generators.p_nom.loc[reserve_i].sum()
p_nom_reserve

The reserve requirement we calculate as the maximum of 20% of the demand or 3 GW:

In [ ]:
reserve_required = n9.loads_t.p_set.sum(axis=1).mul(0.2).clip(lower=3000)
reserve_required.plot()

This is added as a constraint to the model:

In [ ]:
expr = p_nom_reserve - p_reserve >= reserve_required
n9.model.add_constraints(expr, name="reserve_constraints");

### Compound Stress Events

**You decide which events to combine.**

In [ ]:
n10 = base.copy()
n10.name = "compound-event"

## Solving Scenarios

Now, we can solve the different scenarios. Note that the cases where we have added custom constraints do not require to rebuild the model, but just require it to be solved, so the syntax is slightly different for these cases.

In [ ]:
for n in [n0, n1, n2, n3, n4, n5, n6, n7, n10]:
    n.optimize()

n8.optimize.solve_model()
n9.optimize.solve_model()

## Network Collections

PyPSA has a [`pypsa.NetworkCollection`](https://docs.pypsa.org/latest/user-guide/collection/) class that provides a convenient way to manage and analyse multiple networks simultaneously, access their combined data, and generate combined statistics and plots. It uses the names of the networks as indices.

In [ ]:
nc = pypsa.NetworkCollection([n0, n1, n2, n3, n4, n5, n6, n7, n8, n9, n10])

## Evaluation Metrics

### Operational Cost

As absolute values by technology, as total, as relative difference, or as absolute difference to the base case.

In [ ]:
opex = nc.statistics.opex().div(1e6).round(1).droplevel("component").unstack("network")

In [ ]:
opex.sum()

In [ ]:
opex.sum() / opex.sum().min()

In [ ]:
opex.sum() - opex.sum().min()

### Electricity Mix

In [ ]:
energy = (
    nc.statistics.energy_balance()
    .div(1e6)
    .round(1)
    .groupby(["network", "carrier"])
    .sum()
    .unstack("network")
    .drop("AC")
)
energy.T.plot(kind="bar", barmode="stack", labels={"value": "Electricity Supply [TWh]"})

In [ ]:
nc["demand-increase"].statistics.energy_balance.iplot()

### Prices

We can look at average prices or just the frequency of hours with very high prices (e.g. above 100 $/MWh).

In [ ]:
nc.statistics.prices.iplot()

In [ ]:
(nc.buses_t.marginal_price > 100).sum().unstack("network").div(8760 / 8 / 100).round(1)

### Expected Energy Not Served

The expected energy not served (EENS) is a common reliability metric that quantifies the expected amount of energy demand that cannot be met due to supply shortages. It is calculated as the energy used by the load shedding generators in our model.

In [ ]:
# EENS (all scenarios share the same snapshot weightings, so we use the base network's)
(
    n0.snapshot_weightings.generators
    @ nc.generators_t.p.loc[:, nc.generators.carrier == "load-shedding"]
).unstack("network")

### Loss of Load Probability

The loss of load probability (LOLP) is a reliability metric that quantifies the probability that the system will not be able to meet the demand in a given hour. It is calculated as the number of hours with load shedding divided by the total number of hours.

In [ ]:
# LOLP
(
    nc.generators_t.p.loc[:, nc.generators.carrier == "load-shedding"] > 0.1
).sum().unstack("network") / (8760 / 4)

### Maximum Loss of Load

The maximum loss of load (MLOL) is a reliability metric that quantifies the maximum amount of power demand that cannot be met in any given hour. It is calculated as the maximum power used by the load shedding generators in our model.

In [ ]:
nc.generators_t.p.loc[:, nc.generators.carrier == "load-shedding"].max().unstack(
    "network"
)

### Reserve Margin

The reserve margin is a static reliability metric that quantifies the amount of reserve capacity available in the system with the peak demand subtracted. Here, for the base network:

In [ ]:
(
    n0.generators.filter(regex="coal|geothermal|oil|gas|bioenergy", axis=0)
    .groupby("bus")
    .p_nom.sum()
    - n0.loads_t.p_set.max()
)

## Exercises

**Task 1:** For scenario `n10`, create different combinations of the above events and analyse the results. For example, you could combine a drought with a coal import limitation, or a power plant outage with a price shock in oil and gas.

**Task 2:** Change the capacities of the system (e.g. increase solar capacity, add battery storage or increase gas capacity). Repeat the analysis for the different scenarios and compare the results. Can you identify effective strategies for improving the security of supply? Which changes cause more load shedding?